# 🐍 Python Mastery: Basic to Advanced

A comprehensive, hands-on guide covering:
- **Python Basics** — Data structures, comprehensions, generators, decorators, context managers
- **Object-Oriented Programming** — Classes, inheritance, dunder methods, abstract classes, metaclasses
- **Multithreading** — Threads, locks, semaphores, thread pools, GIL
- **Multiprocessing** — Processes, shared memory, process pools, IPC
- **Dynamic Programming** — Memoization, tabulation, classic problems
- **Design Patterns** — Creational, Structural, and Behavioral patterns

---

# 📦 SECTION 1 — Python Basics (Deep Dive)

Even if you know Python well, this section covers the often-overlooked internals and Pythonic patterns.

## 1.1 Data Types & Type System Internals

In [8]:
# Python is dynamically typed but STRONGLY typed.
# Every value is an object. Even int, bool, None.

x = 42
print(type(x))          # <class 'int'>
print(isinstance(x, int))  # True
print(id(x))            # Memory address of the object

# Small integer caching (-5 to 256): CPython caches these objects
a = 256
b = 256
print(a is b)  # True  — same object in memory

c = 257
d = 257
print(c is d)  # False — different objects (outside cache range)
print(c == d)  # True  — same value

<class 'int'>
True
11646664
True
False
True


In [3]:
# String interning
s1 = 'hello'
s2 = 'hello'
print(s1 is s2)   # True — Python interns short strings

s3 = 'hello world!'  # Not interned (has space/special chars)
s4 = 'hello world!'
print(s3 is s4)   # Could be False

# Explicit interning
import sys
s5 = sys.intern('hello world!')
s6 = sys.intern('hello world!')
print(s5 is s6)   # True — now same object

True
False
True


## 1.2 Mutable vs Immutable & Variable Assignment

In [10]:
# Immutable: int, float, str, tuple, frozenset, bytes
# Mutable:   list, dict, set, bytearray, custom objects

# ---- IMMUTABLE ----
a = (1, 2, 3)
try:
    a[0] = 99
except TypeError as e:
    print(f"Tuple error: {e}")

# ---- MUTABLE DEFAULT ARGUMENT TRAP ----
# A classic Python gotcha
def append_item(item, lst=[]):  # BAD: default list is created ONCE
    lst.append(item)
    return lst

print(append_item(1))  # [1]
print(append_item(2))  # [1, 2]  <-- UNEXPECTED!

# CORRECT way:
def append_item_fixed(item, lst=None):
    if lst is None:
        lst = []
    lst.append(item)
    return lst

print(append_item_fixed(1))  # [1]
print(append_item_fixed(2))  # [2]  Correct!

Tuple error: 'tuple' object does not support item assignment
[1]
[1, 2]
[1]
[2]


In [11]:
import copy

original = [[1, 2], [3, 4]]

# Assignment — same object
ref = original
ref[0][0] = 99
print('original after ref change:', original)  # [[99, 2], [3, 4]]

# Shallow copy — new outer list, same inner objects
original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
shallow[0][0] = 99
print('original after shallow change:', original)  # [[99, 2], [3, 4]] — still affected!

# Deep copy — fully independent clone
original = [[1, 2], [3, 4]]
deep = copy.deepcopy(original)
deep[0][0] = 99
print('original after deep change:', original)   # [[1, 2], [3, 4]] — untouched!

original after ref change: [[99, 2], [3, 4]]
original after shallow change: [[99, 2], [3, 4]]
original after deep change: [[1, 2], [3, 4]]


## 1.3 Advanced Data Structures

In [15]:
from collections import (
    Counter, defaultdict, OrderedDict,
    namedtuple, deque, ChainMap
)

# ---- Counter ----
words = ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']
counter = Counter(words)
print(counter)                    # Counter({'apple': 3, 'banana': 2, 'cherry': 1})
print(counter.most_common(2))     # [('apple', 3), ('banana', 2)]

# ---- defaultdict ----
# Avoids KeyError by providing a default factory
graph = defaultdict(list)   # Adjacency list
graph['A'].append('B')
graph['A'].append('C')
print(dict(graph))  # {'A': ['B', 'C']}

# ---- namedtuple ----
# Immutable, memory-efficient alternative to a class
Point = namedtuple('Point', ['x', 'y'])
p = Point(3, 4)
print(p.x, p.y)      # 3 4
print(p._asdict())   # {'x': 3, 'y': 4}

# ---- deque (double-ended queue) ----
# O(1) append/pop from both ends, unlike list which is O(n) at left
dq = deque([1, 2, 3])
dq.appendleft(0)   # O(1)
dq.append(4)       # O(1)
dq.popleft()       # O(1)
print(dq)          # deque([1, 2, 3, 4])

# deque with maxlen acts as a circular buffer
log = deque(maxlen=2)
for i in range(6):
    log.append(i)
print(log)  # deque([3, 4, 5], maxlen=3)

Counter({'apple': 3, 'banana': 2, 'cherry': 1})
[('apple', 3), ('banana', 2)]
{'A': ['B', 'C']}
3 4
{'x': 3, 'y': 4}
deque([1, 2, 3, 4])
deque([4, 5], maxlen=2)


In [ ]:
import heapq
import bisect

# ---- heapq — Min-Heap ----
nums = [5, 1, 8, 3, 2]
heapq.heapify(nums)           # In-place: O(n)
print(heapq.heappop(nums))    # 1 — smallest
heapq.heappush(nums, 0)
print(heapq.heappop(nums))    # 0

# Get N largest/smallest efficiently
data = [3, 1, 4, 1, 5, 9, 2, 6]
print(heapq.nlargest(3, data))   # [9, 6, 5]
print(heapq.nsmallest(3, data))  # [1, 1, 2]

# Max-heap trick: negate the values
max_heap = [-x for x in data]
heapq.heapify(max_heap)
print(-heapq.heappop(max_heap))  # 9 — largest

# ---- bisect — Binary Search on sorted lists ----
sorted_list = [1, 3, 5, 7, 9]
idx = bisect.bisect_left(sorted_list, 5)   # 2 — index where 5 is
idx_right = bisect.bisect_right(sorted_list, 5)  # 3 — after 5
bisect.insort(sorted_list, 6)   # Inserts 6 maintaining sorted order
print(sorted_list)  # [1, 3, 5, 6, 7, 9]

## 1.4 Comprehensions, Generators & Itertools

In [ ]:
# ---- List, Dict, Set Comprehensions ----
squares    = [x**2 for x in range(10) if x % 2 == 0]
square_map = {x: x**2 for x in range(5)}
unique_sq  = {x**2 for x in [-2, -1, 0, 1, 2]}

print(squares)     # [0, 4, 16, 36, 64]
print(square_map)  # {0:0, 1:1, 2:4, 3:9, 4:16}
print(unique_sq)   # {0, 1, 4}

# Nested comprehension — matrix transpose
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transposed = [[row[i] for row in matrix] for i in range(3)]
print(transposed)  # [[1,4,7],[2,5,8],[3,6,9]]

# ---- Generator Expressions ----
# Lazy evaluation — does NOT load all into memory
gen = (x**2 for x in range(1_000_000))  # Uses ~100 bytes!
print(next(gen))  # 0
print(next(gen))  # 1

In [ ]:
# ---- Generator Functions ----
# 'yield' turns a function into a generator

def fibonacci():
    """Infinite Fibonacci sequence generator."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print([next(fib) for _ in range(10)])  # [0,1,1,2,3,5,8,13,21,34]

# ---- yield from ----
# Delegates to a sub-generator
def chain_generators(*iterables):
    for it in iterables:
        yield from it

result = list(chain_generators([1, 2], 'AB', (3, 4)))
print(result)  # [1, 2, 'A', 'B', 3, 4]

# ---- send() — two-way communication with generators ----
def accumulator():
    total = 0
    while True:
        value = yield total   # yield sends total OUT, receives value IN
        if value is None:
            break
        total += value

acc = accumulator()
next(acc)        # Prime the generator (advance to first yield)
print(acc.send(10))   # 10
print(acc.send(20))   # 30
print(acc.send(5))    # 35

In [ ]:
import itertools

# ---- Infinite iterators ----
counter = itertools.count(start=1, step=2)   # 1, 3, 5, 7 ...
cycler  = itertools.cycle('ABC')              # A, B, C, A, B, C ...
repeater = itertools.repeat(42, times=3)      # 42, 42, 42

print(list(itertools.islice(counter, 5)))   # [1, 3, 5, 7, 9]
print(list(itertools.islice(cycler, 6)))    # ['A','B','C','A','B','C']
print(list(repeater))                       # [42, 42, 42]

# ---- Combinatoric generators ----
from itertools import permutations, combinations, combinations_with_replacement, product

print(list(permutations('ABC', 2)))          # All 2-permutations
print(list(combinations('ABC', 2)))          # All 2-combinations
print(list(combinations_with_replacement('AB', 2)))  # With repetition
print(list(product([0,1], repeat=3)))         # Cartesian product (binary 3-bit)

# ---- Groupby ----
data = [('A', 1), ('A', 2), ('B', 3), ('B', 4), ('C', 5)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(key, '->', list(group))

# ---- accumulate, chain, takewhile, dropwhile ----
print(list(itertools.accumulate([1, 2, 3, 4, 5])))          # Running sum: [1,3,6,10,15]
print(list(itertools.chain([1,2], [3,4], [5])))              # [1,2,3,4,5]
print(list(itertools.takewhile(lambda x: x < 4, [1,2,3,4,5])))  # [1,2,3]
print(list(itertools.dropwhile(lambda x: x < 4, [1,2,3,4,5])))  # [4,5]

## 1.5 Decorators

In [ ]:
import functools
import time

# A decorator is a callable that takes a function and returns a new function.

# ---- Basic Decorator ----
def timer(func):
    @functools.wraps(func)  # Preserves the original function's metadata
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))

In [ ]:
# ---- Decorator with Arguments (Decorator Factory) ----
def retry(max_attempts=3, exceptions=(Exception,)):
    """Retries a function on failure up to max_attempts times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
        return wrapper
    return decorator

@retry(max_attempts=3, exceptions=(ValueError,))
def unstable_function(x):
    import random
    if random.random() < 0.7:
        raise ValueError("Random failure!")
    return x * 2

# ---- Class-based Decorator ----
class memoize:
    """Caches function results based on arguments."""
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.cache = {}

    def __call__(self, *args):
        if args not in self.cache:
            self.cache[args] = self.func(*args)
        return self.cache[args]

@memoize
def fib(n):
    if n <= 1: return n
    return fib(n-1) + fib(n-2)

print([fib(i) for i in range(10)])  # [0,1,1,2,3,5,8,13,21,34]

# Python's built-in equivalent:
# @functools.lru_cache(maxsize=None) or @functools.cache (Python 3.9+)

In [ ]:
# ---- Stacking Decorators ----
# Applied bottom-up: @A @B def f → A(B(f))

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello, {name}!"

print(greet('Python'))  # <b><i>Hello, Python!</i></b>

## 1.6 Context Managers

In [ ]:
from contextlib import contextmanager, suppress, ExitStack

# ---- Class-based Context Manager ----
class Timer:
    """Context manager that times a code block."""
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Block took {self.elapsed:.6f}s")
        return False  # Don't suppress exceptions

with Timer():
    total = sum(range(1_000_000))

# ---- Generator-based Context Manager ----
@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    try:
        yield name  # The value bound to 'as' variable
    except Exception as e:
        print(f"Error with {name}: {e}")
        raise
    finally:
        print(f"Releasing {name}")

with managed_resource('DB Connection') as res:
    print(f"Using: {res}")

# ---- suppress — silently ignore exceptions ----
with suppress(FileNotFoundError):
    open('nonexistent_file.txt')
print('Continued after suppressed exception')

# ---- ExitStack — dynamic context managers ----
files = ['file1.txt', 'file2.txt']
# with ExitStack() as stack:
#     handles = [stack.enter_context(open(f, 'w')) for f in files]
#     # All files closed when block exits

## 1.7 Closures, Scopes (LEGB), and `nonlocal`

In [ ]:
# LEGB Rule: Local → Enclosing → Global → Built-in

x = 'global'

def outer():
    x = 'enclosing'
    def inner():
        # x = 'local'  # Would shadow enclosing
        print(x)   # Looks up LEGB: finds 'enclosing'
    inner()

outer()

# ---- Closures ----
def make_counter(start=0):
    count = start
    def increment(step=1):
        nonlocal count  # Modify enclosing scope variable
        count += step
        return count
    return increment

counter = make_counter(10)
print(counter())    # 11
print(counter(5))   # 16
print(counter())    # 17

# Each call to make_counter creates a NEW closure
counter2 = make_counter(100)
print(counter2())   # 101 — independent of counter

# ---- Late binding gotcha in closures ----
# WRONG:
fns_bad = [lambda: i for i in range(5)]
print([f() for f in fns_bad])  # [4,4,4,4,4] — all capture the SAME i

# CORRECT — default argument captures value at definition time:
fns_good = [lambda i=i: i for i in range(5)]
print([f() for f in fns_good])  # [0,1,2,3,4]

## 1.8 Type Hints & Annotations (Python 3.9+)

In [ ]:
from typing import Optional, Union, TypeVar, Generic, Protocol, Literal, TypedDict
from typing import Callable, Iterator, Generator, Any

# ---- Basic annotations ----
def greet(name: str, times: int = 1) -> str:
    return (name + ' ') * times

# ---- Optional, Union ----
def find_user(user_id: int) -> Optional[str]:  # str | None
    db = {1: 'Alice', 2: 'Bob'}
    return db.get(user_id)

def process(value: Union[int, str]) -> str:  # int | str (Python 3.10+)
    return str(value)

# ---- TypeVar — Generic functions ----
T = TypeVar('T')

def first(items: list[T]) -> Optional[T]:
    return items[0] if items else None

# ---- TypedDict ----
class UserInfo(TypedDict):
    name: str
    age: int
    email: Optional[str]

user: UserInfo = {'name': 'Alice', 'age': 30, 'email': None}

# ---- Callable ----
def apply(func: Callable[[int, int], int], a: int, b: int) -> int:
    return func(a, b)

print(apply(lambda x, y: x + y, 3, 4))  # 7

# ---- Protocol (structural subtyping / duck typing) ----
class Drawable(Protocol):
    def draw(self) -> None: ...

class Circle:
    def draw(self) -> None:
        print('Drawing Circle')

def render(shape: Drawable) -> None:
    shape.draw()

render(Circle())  # Works — Circle satisfies Drawable protocol

## 1.9 Exception Handling (Advanced)

In [ ]:
# ---- Custom Exception Hierarchy ----
class AppError(Exception):
    """Base exception for the application."""
    def __init__(self, message: str, code: int = 0):
        super().__init__(message)
        self.code = code

class ValidationError(AppError): pass
class DatabaseError(AppError): pass
class ConnectionError(DatabaseError): pass

def validate_age(age: int):
    if not isinstance(age, int):
        raise ValidationError("Age must be an integer", code=400)
    if age < 0 or age > 150:
        raise ValidationError(f"Age {age} is out of range", code=400)

try:
    validate_age(-5)
except ValidationError as e:
    print(f"Validation failed [{e.code}]: {e}")
except AppError as e:
    print(f"App error: {e}")

# ---- Exception Chaining ----
def read_config(path: str):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError as e:
        raise DatabaseError(f"Config not found: {path}") from e  # Chains exception

# ---- Exception Groups (Python 3.11+) ----
# eg = ExceptionGroup('multiple errors', [ValueError('bad value'), TypeError('wrong type')])
# try:
#     raise eg
# except* ValueError as e:
#     print(f'Handled ValueErrors: {e.exceptions}')
# except* TypeError as e:
#     print(f'Handled TypeErrors: {e.exceptions}')

print('Exception handling examples complete')

## 1.10 Functional Programming Tools

In [ ]:
from functools import reduce, partial, lru_cache, cache
from operator import add, mul

# ---- map, filter, reduce ----
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

doubled  = list(map(lambda x: x * 2, nums))
evens    = list(filter(lambda x: x % 2 == 0, nums))
total    = reduce(add, nums)   # sum
product  = reduce(mul, nums)   # factorial(10)

print(doubled)  # [2,4,6,...,20]
print(evens)    # [2,4,6,8,10]
print(total)    # 55

# ---- partial — freeze arguments ----
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube   = partial(power, exp=3)

print(square(5))  # 25
print(cube(3))    # 27

# ---- lru_cache — Memoization with LRU eviction ----
@lru_cache(maxsize=128)
def expensive_computation(n: int) -> int:
    time.sleep(0.001)  # Simulate expensive work
    return n * n

expensive_computation(10)  # Computed
expensive_computation(10)  # From cache
print(expensive_computation.cache_info())  # CacheInfo(hits=1, misses=1 ...)

---
# 🏛️ SECTION 2 — Object-Oriented Programming (OOP)

Python's OOP is rich: multiple inheritance, MRO, descriptors, metaclasses, and more.

## 2.1 Classes, Attributes, and Methods

In [ ]:
class BankAccount:
    # Class variable — shared across all instances
    interest_rate: float = 0.05
    _account_count: int = 0  # Convention: internal use

    def __init__(self, owner: str, balance: float = 0.0):
        # Instance variables — unique to each instance
        self._owner = owner
        self.__balance = balance  # Name-mangled: _BankAccount__balance
        BankAccount._account_count += 1

    # ---- Instance Method ----
    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount

    def withdraw(self, amount: float) -> None:
        if amount > self.__balance:
            raise ValueError("Insufficient funds")
        self.__balance -= amount

    # ---- Property (computed attribute with validation) ----
    @property
    def balance(self) -> float:
        return self.__balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self.__balance = value

    # ---- Class Method (receives class, not instance) ----
    @classmethod
    def get_account_count(cls) -> int:
        return cls._account_count

    @classmethod
    def from_dict(cls, data: dict) -> 'BankAccount':
        """Alternative constructor (factory method)."""
        return cls(data['owner'], data.get('balance', 0))

    # ---- Static Method (no access to class or instance) ----
    @staticmethod
    def validate_amount(amount: float) -> bool:
        return isinstance(amount, (int, float)) and amount > 0

    def __repr__(self) -> str:
        return f"BankAccount(owner={self._owner!r}, balance={self.__balance:.2f})"

    def __str__(self) -> str:
        return f"{self._owner}'s account: ${self.__balance:.2f}"

acc1 = BankAccount('Alice', 1000)
acc2 = BankAccount.from_dict({'owner': 'Bob', 'balance': 500})

acc1.deposit(250)
print(acc1)                              # Alice's account: $1250.00
print(BankAccount.get_account_count())   # 2
print(repr(acc1))                        # BankAccount(owner='Alice', balance=1250.00)

## 2.2 Dunder (Magic) Methods

In [ ]:
class Vector:
    """2D Vector with full operator support."""
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    # String representations
    def __repr__(self): return f"Vector({self.x}, {self.y})"
    def __str__(self):  return f"({self.x}, {self.y})"

    # Arithmetic operators
    def __add__(self, other): return Vector(self.x + other.x, self.y + other.y)
    def __sub__(self, other): return Vector(self.x - other.x, self.y - other.y)
    def __mul__(self, scalar): return Vector(self.x * scalar, self.y * scalar)
    def __rmul__(self, scalar): return self.__mul__(scalar)  # scalar * v
    def __truediv__(self, scalar): return Vector(self.x / scalar, self.y / scalar)
    def __neg__(self): return Vector(-self.x, -self.y)

    # Comparison operators
    def __eq__(self, other): return self.x == other.x and self.y == other.y
    def __abs__(self): return (self.x**2 + self.y**2) ** 0.5  # magnitude

    # Container protocol
    def __len__(self): return 2
    def __getitem__(self, idx):
        return (self.x, self.y)[idx]
    def __iter__(self):
        yield self.x
        yield self.y

    # Callable
    def __call__(self, scale: float):
        return Vector(self.x * scale, self.y * scale)

v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(v1 + v2)      # (4, 6)
print(v1 * 3)       # (9, 12)
print(3 * v1)       # (9, 12)
print(abs(v1))      # 5.0
print(list(v1))     # [3, 4]
print(v1(0.5))      # (1.5, 2.0)

## 2.3 Inheritance, MRO, and super()

In [ ]:
# ---- Single Inheritance ----
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        raise NotImplementedError  # Template method

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"

class Dog(Animal):
    def speak(self): return f"{self.name} says: Woof!"

class Cat(Animal):
    def speak(self): return f"{self.name} says: Meow!"

# ---- Multiple Inheritance and MRO (C3 Linearization) ----
class Flyable:
    def move(self): return "Flying"

class Swimmable:
    def move(self): return "Swimming"

class Duck(Animal, Flyable, Swimmable):
    def speak(self): return f"{self.name} says: Quack!"
    # move() is inherited from Flyable (leftmost in MRO)

print(Duck.__mro__)  # Shows the method resolution order
duck = Duck('Donald')
print(duck.move())   # Flying — Flyable comes first in MRO

# ---- super() in cooperative multiple inheritance ----
class A:
    def method(self):
        print('A')

class B(A):
    def method(self):
        print('B')
        super().method()  # Calls next in MRO, not necessarily A

class C(A):
    def method(self):
        print('C')
        super().method()

class D(B, C):  # MRO: D -> B -> C -> A
    def method(self):
        print('D')
        super().method()

D().method()  # D, B, C, A — each super() follows the MRO

## 2.4 Abstract Base Classes (ABC) and Interfaces

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class — cannot be instantiated directly."""

    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    # Concrete method available to all subclasses
    def describe(self) -> str:
        return (f"{type(self).__name__}: "
                f"area={self.area():.2f}, perimeter={self.perimeter():.2f}")

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius
    def area(self): return 3.14159 * self.radius ** 2
    def perimeter(self): return 2 * 3.14159 * self.radius

class Rectangle(Shape):
    def __init__(self, w: float, h: float):
        self.w, self.h = w, h
    def area(self): return self.w * self.h
    def perimeter(self): return 2 * (self.w + self.h)

shapes = [Circle(5), Rectangle(4, 6)]
for s in shapes:
    print(s.describe())

# Can't instantiate abstract class:
try:
    Shape()
except TypeError as e:
    print(f"Cannot instantiate: {e}")

## 2.5 Descriptors

In [ ]:
# Descriptors power Python's property, classmethod, staticmethod, functions
# A descriptor defines __get__, __set__, and/or __delete__

class Validated:
    """A descriptor that validates numeric values."""
    def __set_name__(self, owner, name):
        # Called when descriptor is assigned to a class attribute
        self.name = name
        self.storage_name = f'_{owner.__name__}__{name}'

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self  # Accessed via class, not instance
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} must be a number")
        if value < 0:
            raise ValueError(f"{self.name} must be non-negative")
        setattr(obj, self.storage_name, value)

class Product:
    price    = Validated()   # __set_name__ automatically called
    quantity = Validated()

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price         # Calls Validated.__set__
        self.quantity = quantity

    @property
    def total(self):
        return self.price * self.quantity

p = Product('Widget', 9.99, 100)
print(p.total)  # 999.0

try:
    p.price = -5
except ValueError as e:
    print(e)  # price must be non-negative

## 2.6 Metaclasses

In [ ]:
# A metaclass is the 'class of a class'. type is the default metaclass.
# type(name, bases, namespace) dynamically creates a class.

# ---- Creating a class dynamically ----
Dog = type('Dog', (object,), {
    'species': 'Canis lupus familiaris',
    'bark': lambda self: 'Woof!'
})
d = Dog()
print(d.bark())       # Woof!
print(type(Dog))      # <class 'type'>

# ---- Custom Metaclass ----
class SingletonMeta(type):
    """Metaclass that enforces Singleton pattern."""
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=SingletonMeta):
    def __init__(self):
        self.connection = 'Connected'

db1 = Database()
db2 = Database()
print(db1 is db2)  # True — same instance

# ---- __init_subclass__ — simpler alternative to metaclass ----
class PluginBase:
    _plugins = {}

    def __init_subclass__(cls, plugin_name: str = None, **kwargs):
        super().__init_subclass__(**kwargs)
        if plugin_name:
            PluginBase._plugins[plugin_name] = cls

class CSVPlugin(PluginBase, plugin_name='csv'):
    pass

class JSONPlugin(PluginBase, plugin_name='json'):
    pass

print(PluginBase._plugins)  # {'csv': CSVPlugin, 'json': JSONPlugin}

## 2.7 Dataclasses and `__slots__`

In [ ]:
from dataclasses import dataclass, field, asdict, astuple
from typing import ClassVar

@dataclass(order=True, frozen=False)
class Point3D:
    x: float
    y: float
    z: float = 0.0  # Default value
    tags: list = field(default_factory=list)  # Mutable default MUST use field()
    _count: ClassVar[int] = 0  # Class variable, excluded from __init__

    def __post_init__(self):
        # Called after __init__ — for validation or computed fields
        Point3D._count += 1
        object.__setattr__(self, 'magnitude',
                           (self.x**2 + self.y**2 + self.z**2)**0.5)

p1 = Point3D(1, 2, 3)
p2 = Point3D(1, 2, 3)
p3 = Point3D(4, 5, 6, tags=['test'])

print(p1 == p2)      # True — auto __eq__
print(p1 < p3)       # True — auto __lt__ (order=True)
print(asdict(p1))    # {'x': 1, 'y': 2, 'z': 3, 'tags': []}

# ---- __slots__ — Memory optimization ----
# Disables __dict__, stores attributes in fixed-size array
class SlottedPoint:
    __slots__ = ('x', 'y')  # Only these attributes allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y

sp = SlottedPoint(1, 2)
print(sp.x)  # 1
# sp.z = 3   # AttributeError — no __dict__

# Memory comparison (slots use ~40-50% less memory for large collections)
import sys
class RegularPoint:
    def __init__(self, x, y): self.x, self.y = x, y

print(f"Regular: {sys.getsizeof(RegularPoint(1,2))} bytes")
print(f"Slotted: {sys.getsizeof(SlottedPoint(1,2))} bytes")

---
# 🧵 SECTION 3 — Multithreading

Threads share memory space. Best for **I/O-bound** tasks. The **GIL** limits CPU parallelism.

## 3.1 The GIL (Global Interpreter Lock)

In [ ]:
# The GIL is a mutex in CPython that allows only ONE thread to execute Python
# bytecode at a time.
#
# CONSEQUENCE:
#   CPU-bound tasks: Threads do NOT run in parallel → use multiprocessing
#   I/O-bound tasks: GIL is RELEASED during I/O → threads ARE effective
#
# When does the GIL release?
#   - File/network I/O
#   - time.sleep()
#   - C extensions that explicitly release it (numpy, etc.)
#   - After ~100 bytecode instructions (sys.getswitchinterval)

import sys
print(f'Thread switch interval: {sys.getswitchinterval()} seconds')  # 0.005s

# Note: Python 3.13+ has experimental no-GIL (free-threaded) mode!

## 3.2 Creating and Managing Threads

In [ ]:
import threading
import time
import requests  # pip install requests (or use urllib)
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Basic Thread ----
def download_simulation(url: str, delay: float):
    """Simulate downloading a URL."""
    time.sleep(delay)  # GIL released here
    print(f"Downloaded {url} in {delay}s (thread: {threading.current_thread().name})")

# Sequential (total time ≈ sum of all delays)
start = time.perf_counter()
for url, delay in [('url1', 0.5), ('url2', 0.3), ('url3', 0.4)]:
    download_simulation(url, delay)
print(f"Sequential: {time.perf_counter() - start:.2f}s")

# Threaded (total time ≈ max delay)
start = time.perf_counter()
threads = []
for url, delay in [('url1', 0.5), ('url2', 0.3), ('url3', 0.4)]:
    t = threading.Thread(target=download_simulation, args=(url, delay), name=url)
    threads.append(t)
    t.start()

for t in threads:
    t.join()   # Wait for all threads to finish

print(f"Threaded:   {time.perf_counter() - start:.2f}s")

In [ ]:
# ---- Subclassing Thread ----
class WorkerThread(threading.Thread):
    def __init__(self, task_id: int, result_list: list):
        super().__init__(name=f"Worker-{task_id}")
        self.task_id = task_id
        self.result_list = result_list
        self.daemon = True  # Dies when main thread dies

    def run(self):
        # Simulate work
        result = self.task_id ** 2
        time.sleep(0.1)
        self.result_list.append((self.task_id, result))

results = []
workers = [WorkerThread(i, results) for i in range(5)]
for w in workers: w.start()
for w in workers: w.join()

print(sorted(results))  # [(0,0),(1,1),(2,4),(3,9),(4,16)]

## 3.3 Synchronization Primitives

In [ ]:
# ---- Race Condition Demo ----
counter_unsafe = 0

def increment_unsafe():
    global counter_unsafe
    for _ in range(100_000):
        counter_unsafe += 1  # NOT atomic: read-modify-write

threads = [threading.Thread(target=increment_unsafe) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Unsafe counter: {counter_unsafe} (expected 200000)")  # Often wrong!

# ---- Lock — Mutual Exclusion ----
counter_safe = 0
lock = threading.Lock()

def increment_safe():
    global counter_safe
    for _ in range(100_000):
        with lock:  # Acquires lock, ensures mutual exclusion
            counter_safe += 1

threads = [threading.Thread(target=increment_safe) for _ in range(2)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe counter:   {counter_safe} (expected 200000)")  # Always 200000

In [ ]:
# ---- RLock (Reentrant Lock) ----
# A thread can acquire an RLock it already holds
rlock = threading.RLock()

def recursive_task(n):
    with rlock:   # Can be acquired multiple times by same thread
        if n > 0:
            recursive_task(n - 1)

recursive_task(5)  # Would deadlock with regular Lock

# ---- Semaphore — Limit concurrent access ----
# Useful for rate-limiting, connection pools
semaphore = threading.Semaphore(3)  # Max 3 concurrent

def limited_access(thread_id):
    with semaphore:
        print(f"Thread {thread_id} accessing (count: {3 - semaphore._value})")
        time.sleep(0.2)

threads = [threading.Thread(target=limited_access, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()

# ---- Event — Signal between threads ----
ready_event = threading.Event()

def producer():
    time.sleep(0.5)
    print('Producer: data ready')
    ready_event.set()  # Signal all waiting threads

def consumer(name):
    ready_event.wait()  # Block until event is set
    print(f'{name}: consuming data')

threads = [threading.Thread(target=producer),
           threading.Thread(target=consumer, args=('C1',)),
           threading.Thread(target=consumer, args=('C2',))]
for t in threads: t.start()
for t in threads: t.join()

In [ ]:
# ---- Condition Variable — Producer-Consumer Pattern ----
import queue

buffer = queue.Queue(maxsize=5)  # Thread-safe queue

def producer_fn(items):
    for item in items:
        buffer.put(item)  # Blocks if full
        print(f'Produced: {item}')
    buffer.put(None)  # Sentinel to stop consumer

def consumer_fn():
    while True:
        item = buffer.get()  # Blocks if empty
        if item is None:
            break
        print(f'Consumed: {item}')
        buffer.task_done()

p = threading.Thread(target=producer_fn, args=([1, 2, 3, 4, 5],))
c = threading.Thread(target=consumer_fn)
p.start(); c.start()
p.join(); c.join()

# ---- Barrier — Synchronize N threads at a checkpoint ----
barrier = threading.Barrier(3)

def phase_worker(name):
    print(f'{name}: Phase 1 done')
    barrier.wait()  # All 3 must reach here before any proceeds
    print(f'{name}: Phase 2 starting')

threads = [threading.Thread(target=phase_worker, args=(f'W{i}',)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()

## 3.4 ThreadPoolExecutor

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, Future

def fetch_data(task_id: int) -> dict:
    """Simulate an I/O task."""
    time.sleep(0.2)
    return {'id': task_id, 'result': task_id ** 2}

# ---- map — simple parallel map ----
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(fetch_data, range(8)))
print(results[:3])  # [{'id':0,'result':0}, ...]

# ---- submit + as_completed — process results as they finish ----
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_data, i): i for i in range(8)}

    for future in as_completed(futures):
        task_id = futures[future]
        try:
            data = future.result(timeout=1.0)
            print(f"Task {task_id} completed: {data}")
        except Exception as e:
            print(f"Task {task_id} failed: {e}")

# ---- Callbacks ----
def on_done(future: Future):
    print(f"Callback: {future.result()}")

with ThreadPoolExecutor(max_workers=2) as executor:
    f = executor.submit(fetch_data, 42)
    f.add_done_callback(on_done)

---
# ⚙️ SECTION 4 — Multiprocessing

Processes have separate memory spaces. Best for **CPU-bound** tasks. No GIL limitation.

## 4.1 Process Basics

In [ ]:
import multiprocessing as mp
import os

def cpu_bound_task(n: int) -> int:
    """CPU-intensive computation."""
    print(f"PID {os.getpid()} computing {n}")
    return sum(i * i for i in range(n))

# IMPORTANT: On Windows/macOS, multiprocessing code must be in
# if __name__ == '__main__': block. In Jupyter, processes spawn a new interpreter.

if __name__ == '__main__':
    # ---- Basic Process ----
    p = mp.Process(target=cpu_bound_task, args=(1_000_000,))
    p.start()
    p.join()  # Wait for process
    print(f"Process exit code: {p.exitcode}")

    # ---- Multiple Processes ----
    tasks = [500_000, 1_000_000, 750_000, 800_000]
    processes = [mp.Process(target=cpu_bound_task, args=(n,)) for n in tasks]

    start = time.perf_counter()
    for p in processes: p.start()
    for p in processes: p.join()
    print(f"Multiprocessing took: {time.perf_counter() - start:.2f}s")

print("Multiprocessing cells need if __name__ == '__main__' in scripts")

## 4.2 Inter-Process Communication (IPC)

In [ ]:
# ---- Queue — Thread/Process-safe message passing ----
def worker_with_queue(task_queue, result_queue):
    while True:
        task = task_queue.get()
        if task is None:  # Poison pill
            break
        result = task ** 2
        result_queue.put((task, result))

if __name__ == '__main__':
    task_q   = mp.Queue()
    result_q = mp.Queue()

    worker = mp.Process(target=worker_with_queue, args=(task_q, result_q))
    worker.start()

    for i in range(5):
        task_q.put(i)
    task_q.put(None)  # Stop signal

    worker.join()

    results = []
    while not result_q.empty():
        results.append(result_q.get())
    print('IPC Queue results:', sorted(results))

# ---- Pipe — bidirectional channel between 2 processes ----
def pipe_sender(conn):
    data = [1, 2, 3, 4, 5]
    conn.send(data)
    conn.close()

if __name__ == '__main__':
    parent_conn, child_conn = mp.Pipe()
    p = mp.Process(target=pipe_sender, args=(child_conn,))
    p.start()
    received = parent_conn.recv()
    p.join()
    print('Pipe received:', received)

## 4.3 Shared Memory and Synchronization

In [ ]:
# ---- Value and Array — shared memory objects ----
def increment_shared(val, lock):
    for _ in range(100_000):
        with lock:
            val.value += 1

if __name__ == '__main__':
    shared_val = mp.Value('i', 0)  # 'i' = C int
    mp_lock    = mp.Lock()

    procs = [mp.Process(target=increment_shared, args=(shared_val, mp_lock))
             for _ in range(2)]
    for p in procs: p.start()
    for p in procs: p.join()
    print(f"Shared counter: {shared_val.value}")  # 200000

# ---- multiprocessing.shared_memory (Python 3.8+) ----
import numpy as np
from multiprocessing import shared_memory

def worker_shared_mem(shm_name, shape, dtype):
    existing_shm = shared_memory.SharedMemory(name=shm_name)
    arr = np.ndarray(shape, dtype=dtype, buffer=existing_shm.buf)
    arr *= 2  # Modify in place
    existing_shm.close()

if __name__ == '__main__':
    data = np.array([1, 2, 3, 4, 5], dtype=np.int64)
    shm = shared_memory.SharedMemory(create=True, size=data.nbytes)
    shared_arr = np.ndarray(data.shape, dtype=data.dtype, buffer=shm.buf)
    shared_arr[:] = data[:]

    p = mp.Process(target=worker_shared_mem, args=(shm.name, data.shape, data.dtype))
    p.start()
    p.join()

    print('After worker:', shared_arr)  # [2, 4, 6, 8, 10]
    shm.close()
    shm.unlink()

## 4.4 ProcessPoolExecutor

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def is_prime(n: int) -> bool:
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0: return False
    return True

numbers = range(100_000, 100_100)

if __name__ == '__main__':
    # Sequential
    start = time.perf_counter()
    seq_primes = [n for n in numbers if is_prime(n)]
    print(f'Sequential: {time.perf_counter()-start:.3f}s → {len(seq_primes)} primes')

    # Parallel (uses all CPU cores)
    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=mp.cpu_count()) as executor:
        # chunksize batches tasks to reduce IPC overhead
        results = list(executor.map(is_prime, numbers, chunksize=10))
    par_primes = [n for n, prime in zip(numbers, results) if prime]
    print(f'Parallel:   {time.perf_counter()-start:.3f}s → {len(par_primes)} primes')
    print(f'CPU count:  {mp.cpu_count()}')

## 4.5 `asyncio` — Cooperative Concurrency

In [ ]:
import asyncio

# asyncio is single-threaded but concurrent via cooperative scheduling.
# Best for high-concurrency I/O (thousands of connections).

async def fetch_async(url: str, delay: float) -> str:
    """Simulate async HTTP request."""
    await asyncio.sleep(delay)   # Non-blocking sleep
    return f"Data from {url}"

async def main():
    urls = [('api/users', 0.3), ('api/posts', 0.5), ('api/comments', 0.2)]

    # gather — run all concurrently
    start = time.perf_counter()
    results = await asyncio.gather(*[fetch_async(u, d) for u, d in urls])
    print(f"gather took: {time.perf_counter()-start:.2f}s")
    print(results)

    # TaskGroup (Python 3.11+)
    # async with asyncio.TaskGroup() as tg:
    #     tasks = [tg.create_task(fetch_async(u, d)) for u, d in urls]

await main()  # In Jupyter, top-level await works

In [ ]:
# ---- Async Context Managers and Generators ----
class AsyncDBConnection:
    async def __aenter__(self):
        await asyncio.sleep(0.01)  # Simulate connection
        print('DB Connected')
        return self

    async def __aexit__(self, *args):
        await asyncio.sleep(0.01)  # Simulate disconnect
        print('DB Disconnected')

    async def query(self, sql: str):
        await asyncio.sleep(0.05)
        return f"Results for: {sql}"

async def async_generator_example():
    async def async_range(n):
        for i in range(n):
            await asyncio.sleep(0.01)
            yield i

    async for item in async_range(5):
        print(f'async item: {item}')

async def run_all():
    async with AsyncDBConnection() as db:
        result = await db.query('SELECT * FROM users')
        print(result)

    await async_generator_example()

    # asyncio.Queue for async producer-consumer
    q = asyncio.Queue(maxsize=3)

    async def producer():
        for i in range(5):
            await q.put(i)
            print(f'Async produced: {i}')

    async def consumer():
        for _ in range(5):
            item = await q.get()
            print(f'Async consumed: {item}')
            q.task_done()

    await asyncio.gather(producer(), consumer())

await run_all()

---
# 🧮 SECTION 5 — Dynamic Programming (DP)

DP solves optimization problems by breaking them into overlapping subproblems and storing results.

## 5.1 Memoization (Top-Down DP)

In [ ]:
from functools import lru_cache
import sys
sys.setrecursionlimit(10_000)

# ---- Fibonacci ----
# Naive: O(2^n), Memoized: O(n)

@lru_cache(maxsize=None)
def fib_memo(n: int) -> int:
    if n <= 1: return n
    return fib_memo(n-1) + fib_memo(n-2)

print([fib_memo(i) for i in range(15)])  # [0,1,1,2,3,5,8,...]

# ---- Staircase Problem ----
# How many ways to climb n stairs taking 1 or 2 steps at a time?
@lru_cache(maxsize=None)
def climb_stairs(n: int) -> int:
    if n <= 2: return n
    return climb_stairs(n-1) + climb_stairs(n-2)

for n in [1, 2, 3, 4, 5, 10]:
    print(f"climb_stairs({n}) = {climb_stairs(n)}")

# ---- Coin Change (min coins) ----
@lru_cache(maxsize=None)
def coin_change_memo(coins: tuple, amount: int) -> int:
    """
    Returns minimum coins to make 'amount', or -1 if impossible.
    coins must be a tuple (hashable) for lru_cache.
    """
    if amount == 0: return 0
    if amount < 0:  return float('inf')

    min_coins = float('inf')
    for coin in coins:
        sub = coin_change_memo(coins, amount - coin)
        min_coins = min(min_coins, sub + 1)

    return min_coins if min_coins != float('inf') else -1

coins = (1, 5, 10, 25)
print(f"Min coins for 30: {coin_change_memo(coins, 30)}")  # 2 (25+5)

## 5.2 Tabulation (Bottom-Up DP)

In [ ]:
# Bottom-up: fill a table iteratively, avoids recursion overhead and stack overflow

# ---- 0/1 Knapsack ----
# Given items with weights/values, maximize value within weight capacity

def knapsack(weights: list, values: list, capacity: int) -> int:
    n = len(weights)
    # dp[i][w] = max value using first i items with weight limit w
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(capacity + 1):
            # Don't take item i
            dp[i][w] = dp[i-1][w]
            # Take item i (if it fits)
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w],
                               dp[i-1][w - weights[i-1]] + values[i-1])

    return dp[n][capacity]

weights = [2, 3, 4, 5]
values  = [3, 4, 5, 6]
cap     = 8
print(f"Knapsack max value: {knapsack(weights, values, cap)}")  # 10

# Space-optimized version (O(W) space instead of O(N*W))
def knapsack_optimized(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for i in range(len(weights)):
        for w in range(capacity, weights[i] - 1, -1):  # Reverse to avoid reuse
            dp[w] = max(dp[w], dp[w - weights[i]] + values[i])
    return dp[capacity]

print(f"Knapsack optimized: {knapsack_optimized(weights, values, cap)}")  # 10

In [ ]:
# ---- Longest Common Subsequence (LCS) ----
def lcs(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[m][n]

print(f"LCS('ABCBDAB', 'BDCAB') = {lcs('ABCBDAB', 'BDCAB')}")  # 4 (BCAB or BDAB)

# ---- Longest Increasing Subsequence (LIS) — O(n log n) ----
def lis(nums: list) -> int:
    tails = []  # tails[i] = smallest tail of LIS of length i+1
    for num in nums:
        lo, hi = 0, len(tails)
        while lo < hi:  # Binary search for insertion point
            mid = (lo + hi) // 2
            if tails[mid] < num:
                lo = mid + 1
            else:
                hi = mid
        if lo == len(tails):
            tails.append(num)
        else:
            tails[lo] = num
    return len(tails)

arr = [10, 9, 2, 5, 3, 7, 101, 18]
print(f"LIS length: {lis(arr)}")  # 4 (2,3,7,101 or 2,5,7,101)

In [ ]:
# ---- Edit Distance (Levenshtein) ----
def edit_distance(s1: str, s2: str) -> int:
    """Minimum insert/delete/replace operations to transform s1 into s2."""
    m, n = len(s1), len(s2)
    # dp[i][j] = edit distance between s1[:i] and s2[:j]
    dp = list(range(n + 1))  # Base case: s1 is empty

    for i in range(1, m + 1):
        new_dp = [i] + [0] * n   # i deletions from s1
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                new_dp[j] = dp[j-1]  # No operation needed
            else:
                new_dp[j] = 1 + min(dp[j],      # Delete
                                    new_dp[j-1], # Insert
                                    dp[j-1])     # Replace
        dp = new_dp

    return dp[n]

print(f"Edit distance('kitten', 'sitting') = {edit_distance('kitten', 'sitting')}")  # 3

# ---- Matrix Chain Multiplication ----
def matrix_chain(dims: list) -> int:
    """Minimum scalar multiplications to multiply a chain of matrices."""
    n = len(dims) - 1  # Number of matrices
    dp = [[0] * n for _ in range(n)]

    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = (dp[i][k] + dp[k+1][j] +
                        dims[i] * dims[k+1] * dims[j+1])
                dp[i][j] = min(dp[i][j], cost)

    return dp[0][n-1]

# Matrices: 10x30, 30x5, 5x60
dims = [10, 30, 5, 60]
print(f"Matrix chain min cost: {matrix_chain(dims)}")  # 4500

In [ ]:
# ---- Unique Paths in Grid ----
def unique_paths(m: int, n: int) -> int:
    """Count paths from top-left to bottom-right (only right/down moves)."""
    dp = [1] * n  # Top row: all 1s
    for _ in range(1, m):
        for j in range(1, n):
            dp[j] += dp[j-1]
    return dp[-1]

print(f"Unique paths 3x7: {unique_paths(3, 7)}")  # 28

# ---- Partition Equal Subset Sum ----
def can_partition(nums: list) -> bool:
    """Can nums be split into two subsets with equal sum?"""
    total = sum(nums)
    if total % 2 != 0: return False
    target = total // 2

    dp = {0}  # Set of achievable sums
    for num in nums:
        dp = {s + num for s in dp} | dp

    return target in dp

print(can_partition([1, 5, 11, 5]))  # True  (1+5+5 == 11)
print(can_partition([1, 2, 3, 5]))   # False

# ---- Word Break ----
def word_break(s: str, word_dict: list) -> bool:
    """Can 's' be segmented into words from word_dict?"""
    word_set = set(word_dict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True

    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break

    return dp[n]

print(word_break('leetcode', ['leet', 'code']))   # True
print(word_break('applepenapple', ['apple', 'pen']))  # True

---
# 🏗️ SECTION 6 — Design Patterns

Design patterns are reusable solutions to common software design problems.

## 6.1 Creational Patterns

In [ ]:
# ════════════════════════════════════════════════
# SINGLETON — ensure only one instance exists
# ════════════════════════════════════════════════
import threading

class Singleton:
    _instance = None
    _lock = threading.Lock()  # Thread-safe instantiation

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:  # Double-checked locking
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value: int = 0):
        if not hasattr(self, '_initialized'):
            self.value = value
            self._initialized = True

s1 = Singleton(42)
s2 = Singleton(99)  # Returns existing instance
print(s1 is s2)     # True
print(s1.value)     # 42 — __init__ only runs once

In [ ]:
# ════════════════════════════════════════════════
# FACTORY METHOD — delegate object creation to subclasses
# ════════════════════════════════════════════════
from abc import ABC, abstractmethod

class Notification(ABC):
    @abstractmethod
    def send(self, message: str) -> str: ...

class EmailNotification(Notification):
    def send(self, message): return f"Email: {message}"

class SMSNotification(Notification):
    def send(self, message): return f"SMS: {message}"

class PushNotification(Notification):
    def send(self, message): return f"Push: {message}"

class NotificationFactory:
    _registry = {
        'email': EmailNotification,
        'sms':   SMSNotification,
        'push':  PushNotification,
    }

    @classmethod
    def create(cls, kind: str) -> Notification:
        if kind not in cls._registry:
            raise ValueError(f"Unknown notification type: {kind}")
        return cls._registry[kind]()

    @classmethod
    def register(cls, kind: str, cls_type):
        """Extensible — register new types at runtime."""
        cls._registry[kind] = cls_type

for kind in ['email', 'sms', 'push']:
    n = NotificationFactory.create(kind)
    print(n.send('Hello!'))

In [ ]:
# ════════════════════════════════════════════════
# ABSTRACT FACTORY — family of related objects
# ════════════════════════════════════════════════
class Button(ABC):
    @abstractmethod
    def render(self) -> str: ...

class Checkbox(ABC):
    @abstractmethod
    def render(self) -> str: ...

class WindowsButton(Button):
    def render(self): return "[Windows Button]"

class WindowsCheckbox(Checkbox):
    def render(self): return "[Windows Checkbox]"

class MacButton(Button):
    def render(self): return "(Mac Button)"

class MacCheckbox(Checkbox):
    def render(self): return "(Mac Checkbox)"

class GUIFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...
    @abstractmethod
    def create_checkbox(self) -> Checkbox: ...

class WindowsFactory(GUIFactory):
    def create_button(self):   return WindowsButton()
    def create_checkbox(self): return WindowsCheckbox()

class MacFactory(GUIFactory):
    def create_button(self):   return MacButton()
    def create_checkbox(self): return MacCheckbox()

def build_ui(factory: GUIFactory):
    btn = factory.create_button()
    chk = factory.create_checkbox()
    print(btn.render(), chk.render())

build_ui(WindowsFactory())
build_ui(MacFactory())

# ════════════════════════════════════════════════
# BUILDER — construct complex objects step by step
# ════════════════════════════════════════════════
class QueryBuilder:
    def __init__(self):
        self._table  = ''
        self._fields = ['*']
        self._where  = []
        self._limit  = None
        self._order  = None

    def from_table(self, table: str) -> 'QueryBuilder':
        self._table = table
        return self  # Return self for chaining

    def select(self, *fields) -> 'QueryBuilder':
        self._fields = list(fields)
        return self

    def where(self, condition: str) -> 'QueryBuilder':
        self._where.append(condition)
        return self

    def order_by(self, field: str, desc=False) -> 'QueryBuilder':
        self._order = f"{field} {'DESC' if desc else 'ASC'}"
        return self

    def limit(self, n: int) -> 'QueryBuilder':
        self._limit = n
        return self

    def build(self) -> str:
        query = f"SELECT {', '.join(self._fields)} FROM {self._table}"
        if self._where:
            query += f" WHERE {' AND '.join(self._where)}"
        if self._order:
            query += f" ORDER BY {self._order}"
        if self._limit:
            query += f" LIMIT {self._limit}"
        return query

query = (QueryBuilder()
         .from_table('users')
         .select('id', 'name', 'email')
         .where('age > 18')
         .where('active = 1')
         .order_by('name')
         .limit(10)
         .build())
print(query)

In [ ]:
# ════════════════════════════════════════════════
# PROTOTYPE — clone existing objects
# ════════════════════════════════════════════════
import copy

class Prototype:
    def clone(self):
        return copy.deepcopy(self)

class GameCharacter(Prototype):
    def __init__(self, name, health, skills):
        self.name   = name
        self.health = health
        self.skills = skills  # Mutable list

    def __repr__(self):
        return f"GameCharacter({self.name}, hp={self.health}, skills={self.skills})"

hero = GameCharacter('Hero', 100, ['fire', 'ice'])
hero_clone = hero.clone()
hero_clone.name = 'Evil Clone'
hero_clone.skills.append('poison')

print(hero)        # Original unaffected (deep copy)
print(hero_clone)  # Clone has extra skill

## 6.2 Structural Patterns

In [ ]:
# ════════════════════════════════════════════════
# ADAPTER — make incompatible interfaces work together
# ════════════════════════════════════════════════
class EuropeanSocket:
    def plug_in(self, voltage=220): return f"European: {voltage}V"

class AmericanDevice:
    def connect(self, voltage=110): return f"American device needs {voltage}V"

class SocketAdapter:
    """Adapts European socket to work with American device."""
    def __init__(self, european_socket: EuropeanSocket):
        self._socket = european_socket

    def connect(self, voltage=110):
        # Adapt the call
        converted = self._socket.plug_in(220)
        return f"Adapter converts 220V→110V. {converted} → {voltage}V output"

adapter = SocketAdapter(EuropeanSocket())
device  = AmericanDevice()
print(device.connect())   # Without adapter
print(adapter.connect())  # With adapter

# ════════════════════════════════════════════════
# DECORATOR PATTERN — add behaviour without subclassing
# ════════════════════════════════════════════════
class Coffee(ABC):
    @abstractmethod
    def cost(self) -> float: ...
    @abstractmethod
    def description(self) -> str: ...

class SimpleCoffee(Coffee):
    def cost(self): return 1.00
    def description(self): return 'Simple coffee'

class CoffeeDecorator(Coffee):
    def __init__(self, coffee: Coffee):
        self._coffee = coffee
    def cost(self): return self._coffee.cost()
    def description(self): return self._coffee.description()

class MilkDecorator(CoffeeDecorator):
    def cost(self): return self._coffee.cost() + 0.25
    def description(self): return self._coffee.description() + ', milk'

class SugarDecorator(CoffeeDecorator):
    def cost(self): return self._coffee.cost() + 0.10
    def description(self): return self._coffee.description() + ', sugar'

class WhipDecorator(CoffeeDecorator):
    def cost(self): return self._coffee.cost() + 0.50
    def description(self): return self._coffee.description() + ', whip'

coffee = WhipDecorator(SugarDecorator(MilkDecorator(SimpleCoffee())))
print(f"{coffee.description()} → ${coffee.cost():.2f}")

In [ ]:
# ════════════════════════════════════════════════
# PROXY — control access to an object
# ════════════════════════════════════════════════
class RealDatabase:
    def __init__(self):
        print("Expensive DB initialization...")

    def query(self, sql: str) -> str:
        return f"Results: {sql}"

class DatabaseProxy:
    """Lazy-loading + caching proxy."""
    def __init__(self):
        self._db = None   # Not initialized yet (lazy)
        self._cache = {}

    def query(self, sql: str) -> str:
        if sql in self._cache:
            print(f"[Cache hit] {sql}")
            return self._cache[sql]

        if self._db is None:     # Lazy initialization
            self._db = RealDatabase()

        result = self._db.query(sql)
        self._cache[sql] = result
        return result

db = DatabaseProxy()
print(db.query('SELECT * FROM users'))   # Initializes DB
print(db.query('SELECT * FROM users'))   # From cache

# ════════════════════════════════════════════════
# FACADE — simplified interface to a complex system
# ════════════════════════════════════════════════
class VideoDecoder:
    def decode(self, file): return f"Decoded {file}"

class AudioMixer:
    def mix(self, audio): return f"Mixed {audio}"

class Subtitles:
    def load(self, lang): return f"Loaded {lang} subtitles"

class VideoPlayerFacade:
    """Simple interface to complex video subsystem."""
    def __init__(self):
        self._decoder   = VideoDecoder()
        self._mixer     = AudioMixer()
        self._subtitles = Subtitles()

    def play(self, filename: str, lang: str = 'en') -> None:
        print(self._decoder.decode(filename))
        print(self._mixer.mix('stereo'))
        print(self._subtitles.load(lang))
        print(f"Playing: {filename}")

player = VideoPlayerFacade()
player.play('movie.mp4', 'fr')

In [ ]:
# ════════════════════════════════════════════════
# COMPOSITE — treat individual objects and groups uniformly
# ════════════════════════════════════════════════
class FileSystemComponent(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def size(self) -> int: ...

    def display(self, indent=0):
        print(' ' * indent + self.name)

class File(FileSystemComponent):
    def __init__(self, name, size):
        super().__init__(name)
        self._size = size

    def size(self): return self._size

class Folder(FileSystemComponent):
    def __init__(self, name):
        super().__init__(name)
        self._children = []

    def add(self, component): self._children.append(component); return self

    def size(self): return sum(c.size() for c in self._children)

    def display(self, indent=0):
        print(' ' * indent + f"📁 {self.name}/")
        for child in self._children:
            child.display(indent + 2)

root = Folder('root')
docs = Folder('docs')
docs.add(File('resume.pdf', 120)).add(File('cover.docx', 85))
root.add(docs).add(File('readme.md', 10))

root.display()
print(f"Total size: {root.size()} KB")

# ════════════════════════════════════════════════
# FLYWEIGHT — share state to support many fine-grained objects
# ════════════════════════════════════════════════
class CharacterGlyph:
    """Shared intrinsic state (font, size)."""
    def __init__(self, char: str, font: str, size: int):
        self.char = char
        self.font = font
        self.size = size

    def render(self, x: int, y: int):
        return f"Render '{self.char}'({self.font},{self.size}) at ({x},{y})"

class GlyphFactory:
    _glyphs = {}  # Shared pool

    @classmethod
    def get_glyph(cls, char, font, size) -> CharacterGlyph:
        key = (char, font, size)
        if key not in cls._glyphs:
            cls._glyphs[key] = CharacterGlyph(char, font, size)
        return cls._glyphs[key]

# Extrinsic state (position) is stored outside
text = [(c, i*10) for i, c in enumerate('HELLO')]
for char, x in text:
    glyph = GlyphFactory.get_glyph(char, 'Arial', 12)
    print(glyph.render(x, 0))

print(f"Unique glyphs created: {len(GlyphFactory._glyphs)}")

## 6.3 Behavioral Patterns

In [ ]:
# ════════════════════════════════════════════════
# OBSERVER — notify dependents of state changes
# ════════════════════════════════════════════════
from typing import Protocol

class Observer(Protocol):
    def update(self, event: str, data: any) -> None: ...

class EventBus:
    """Generic publish-subscribe event bus."""
    def __init__(self):
        self._subscribers: dict[str, list] = {}

    def subscribe(self, event: str, handler) -> None:
        self._subscribers.setdefault(event, []).append(handler)

    def unsubscribe(self, event: str, handler) -> None:
        if event in self._subscribers:
            self._subscribers[event].remove(handler)

    def publish(self, event: str, data=None) -> None:
        for handler in self._subscribers.get(event, []):
            handler(event, data)

bus = EventBus()

def email_handler(event, data): print(f"Email: {event} → {data}")
def log_handler(event, data):   print(f"Log:   {event} → {data}")
def sms_handler(event, data):   print(f"SMS:   {event} → {data}")

bus.subscribe('user.registered', email_handler)
bus.subscribe('user.registered', log_handler)
bus.subscribe('order.placed',    sms_handler)
bus.subscribe('order.placed',    email_handler)

bus.publish('user.registered', {'name': 'Alice', 'email': 'alice@example.com'})
bus.publish('order.placed',    {'order_id': 42, 'amount': 99.99})

In [ ]:
# ════════════════════════════════════════════════
# STRATEGY — swap algorithms at runtime
# ════════════════════════════════════════════════
from typing import Callable

# Functional style — strategies as callables
SortStrategy = Callable[[list], list]

def bubble_sort(data: list) -> list:
    arr = data.copy()
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

def merge_sort(data: list) -> list:
    if len(data) <= 1: return data
    mid = len(data) // 2
    left  = merge_sort(data[:mid])
    right = merge_sort(data[mid:])
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i += 1
        else:                    result.append(right[j]); j += 1
    return result + left[i:] + right[j:]

class Sorter:
    def __init__(self, strategy: SortStrategy = sorted):
        self.strategy = strategy

    def sort(self, data: list) -> list:
        return self.strategy(data)

data = [64, 34, 25, 12, 22, 11, 90]
sorter = Sorter(merge_sort)
print(sorter.sort(data))

sorter.strategy = bubble_sort  # Swap strategy at runtime
print(sorter.sort(data))

# ════════════════════════════════════════════════
# COMMAND — encapsulate operations as objects
# ════════════════════════════════════════════════
class Command(ABC):
    @abstractmethod
    def execute(self) -> None: ...
    @abstractmethod
    def undo(self) -> None: ...

class TextEditor:
    def __init__(self): self.text = ''
    def insert(self, s): self.text += s
    def delete(self, n): self.text = self.text[:-n]

class InsertCommand(Command):
    def __init__(self, editor: TextEditor, text: str):
        self.editor = editor
        self.text = text

    def execute(self): self.editor.insert(self.text)
    def undo(self): self.editor.delete(len(self.text))

class CommandHistory:
    def __init__(self): self._history: list[Command] = []

    def execute(self, cmd: Command):
        cmd.execute()
        self._history.append(cmd)

    def undo(self):
        if self._history:
            self._history.pop().undo()

editor  = TextEditor()
history = CommandHistory()

history.execute(InsertCommand(editor, 'Hello'))
history.execute(InsertCommand(editor, ', World'))
print(editor.text)   # Hello, World
history.undo()
print(editor.text)   # Hello
history.undo()
print(editor.text)   # (empty)

In [ ]:
# ════════════════════════════════════════════════
# ITERATOR — custom traversal of a collection
# ════════════════════════════════════════════════
class BinaryTree:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left  = left
        self.right = right

    def __iter__(self):
        """In-order traversal (Left → Root → Right)."""
        if self.left:  yield from self.left   # Recursively iterate
        yield self.value
        if self.right: yield from self.right

tree = BinaryTree(4,
           BinaryTree(2, BinaryTree(1), BinaryTree(3)),
           BinaryTree(6, BinaryTree(5), BinaryTree(7)))

print(list(tree))  # [1,2,3,4,5,6,7] — sorted!

# ════════════════════════════════════════════════
# TEMPLATE METHOD — define skeleton, defer steps to subclasses
# ════════════════════════════════════════════════
class DataProcessor(ABC):
    def process(self, data) -> any:
        """Template method — fixed algorithm skeleton."""
        raw     = self.read(data)
        parsed  = self.parse(raw)
        result  = self.transform(parsed)
        self.save(result)
        return result

    def read(self, data): return data              # Default: pass-through
    @abstractmethod
    def parse(self, raw): ...
    @abstractmethod
    def transform(self, parsed): ...
    def save(self, result): print(f"Saved: {result}")  # Default: print

class CSVProcessor(DataProcessor):
    def parse(self, raw):
        return [row.split(',') for row in raw.split('\n') if row]
    def transform(self, parsed):
        return [dict(zip(parsed[0], row)) for row in parsed[1:]]

csv_data = "name,age\nAlice,30\nBob,25"
result = CSVProcessor().process(csv_data)
print(result)

# ════════════════════════════════════════════════
# CHAIN OF RESPONSIBILITY — pass request along handler chain
# ════════════════════════════════════════════════
class Handler(ABC):
    def __init__(self):
        self._next: 'Handler' = None

    def set_next(self, handler: 'Handler') -> 'Handler':
        self._next = handler
        return handler  # For fluent chaining

    def handle(self, request):
        if self._next:
            return self._next.handle(request)
        return None

class AuthHandler(Handler):
    def handle(self, request):
        if not request.get('authenticated'):
            return 'Error: Not authenticated'
        return super().handle(request)

class RateLimitHandler(Handler):
    def handle(self, request):
        if request.get('rate_limit_exceeded'):
            return 'Error: Rate limit exceeded'
        return super().handle(request)

class BusinessLogicHandler(Handler):
    def handle(self, request):
        return f"Processed: {request['data']}"

# Build the chain
auth   = AuthHandler()
rate   = RateLimitHandler()
logic  = BusinessLogicHandler()
auth.set_next(rate).set_next(logic)

for req in [
    {'authenticated': False, 'data': 'x'},
    {'authenticated': True, 'rate_limit_exceeded': True, 'data': 'y'},
    {'authenticated': True, 'rate_limit_exceeded': False, 'data': 'z'},
]:
    print(auth.handle(req))

In [ ]:
# ════════════════════════════════════════════════
# STATE — change behaviour based on internal state
# ════════════════════════════════════════════════
class TrafficLight:
    class State(ABC):
        @abstractmethod
        def next_state(self, light) -> None: ...
        @abstractmethod
        def display(self) -> str: ...

    class RedState(State):
        def display(self): return '🔴 STOP'
        def next_state(self, light): light._state = TrafficLight.GreenState()

    class GreenState(State):
        def display(self): return '🟢 GO'
        def next_state(self, light): light._state = TrafficLight.YellowState()

    class YellowState(State):
        def display(self): return '🟡 CAUTION'
        def next_state(self, light): light._state = TrafficLight.RedState()

    def __init__(self):
        self._state = self.RedState()

    def tick(self):
        print(self._state.display())
        self._state.next_state(self)

light = TrafficLight()
for _ in range(6):
    light.tick()

# ════════════════════════════════════════════════
# MEMENTO — capture and restore state (undo/redo)
# ════════════════════════════════════════════════
from dataclasses import dataclass

@dataclass(frozen=True)  # Immutable snapshot
class Memento:
    state: dict

class GameState:
    def __init__(self):
        self._state = {'level': 1, 'hp': 100, 'score': 0}
        self._history = []

    def update(self, **kwargs):
        self._history.append(Memento(dict(self._state)))
        self._state.update(kwargs)

    def undo(self):
        if self._history:
            self._state = dict(self._history.pop().state)

    def __repr__(self): return f"GameState({self._state})"

game = GameState()
game.update(level=2, score=500)
game.update(hp=50, score=1000)
print(game)     # level=2, hp=50, score=1000
game.undo()
print(game)     # level=2, hp=100, score=500
game.undo()
print(game)     # level=1, hp=100, score=0

---
## 📚 Summary

| Section | Key Concepts |
|---------|-------------|
| **Python Basics** | Type system, mutability, comprehensions, generators, decorators, context managers, closures, type hints |
| **OOP** | Classes, dunder methods, MRO, ABCs, descriptors, metaclasses, dataclasses |
| **Multithreading** | GIL, Thread, Lock, Semaphore, Event, Barrier, Queue, ThreadPoolExecutor, asyncio |
| **Multiprocessing** | Process, IPC (Queue/Pipe), shared memory, ProcessPoolExecutor |
| **Dynamic Programming** | Memoization, tabulation, Knapsack, LCS, LIS, Edit Distance, Matrix Chain |
| **Design Patterns** | Singleton, Factory, Builder, Prototype, Adapter, Decorator, Proxy, Facade, Observer, Strategy, Command, State, Memento |

### Further Reading
- [Python Docs](https://docs.python.org/3/)
- [Real Python](https://realpython.com/)
- [Fluent Python (O'Reilly)](https://www.oreilly.com/library/view/fluent-python-2nd/9781492056348/)
- [Python Cookbook (O'Reilly)](https://www.oreilly.com/library/view/python-cookbook-3rd/9781449357337/)